# 03 — Preprocessing experiments

Compare bounded preprocessing hypotheses with native five-class detection evidence. Raw/Skip wins ties; no filter is mandatory. This notebook selects only a calibrated pretrained candidate, never a post-fine-tuning winner.

**Inputs:** Frozen Round 2 audit/scenarios, original BDD sources, frozen v2 and pinned YOLO26n.

**Outputs:** Immutable experiment plan, native metrics/matches, subgroup/error/cost reports, an exploration shortlist and a confirmation decision. A successful confirmation writes a versioned policy config/hash for #33.

**Prerequisites:** Saved notebooks 01/02 artifacts and study kernel. No upstream notebook, training or final-holdout access. Default is saved-result reuse; set `ALLOW_INFERENCE=True` to measure missing comparisons.

**Outline**

- [Configuration and hypothesis](#configuration)
- [Individual operators](#individual-operators)
- [Composition and order](#composition-order)
- [Adaptive comparison](#adaptive-comparison)
- [Selected policy and confirmation](#selected-policy)
- [Summary and evidence](#summary)

<a id="configuration"></a>

## 1. Configuration and hypothesis

Hypothesis: stronger or combined filters may harm AP even when images look better. Screen two strengths for Gamma/CLAHE/Denoise/Sharpen and reuse two alternatives: [bilateral filtering (Tomasi & Manduchi)](https://homepages.inf.ed.ac.uk/rbf/CVonline/LOCAL_COPIES/MANDUCHI1/Bilateral_Filtering.html) and luminance percentile stretching, inspired by [Simplest Color Balance (Limare et al.)](https://www.ipol.im/pub/art/2011/llmps-scb/). The latter is the existing LAB-luminance adaptation, not the paper's independent RGB-channel method. These papers motivate hypotheses, not detection gains.

Use complete source groups sampled separately inside the saved calibration halves. All candidates use the same IDs, five-class GT, unchanged detector/settings and a single JPEG95 encode after corruption/treatment. Fixed quality thresholds are preregistered hypotheses; routing cannot see synthetic labels. The score gives equal weight to clean, low-light, noise and motion families, and equal weight to severities within each synthetic family. Fewer than five source groups for any target class blocks selection. The support audit expanded the initial 128-image plan to 256 before new AP measurements; increase the sample in a **new output root** before making scientific claims.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from vehicle_detection.adaptive import validate_policy
from vehicle_detection.baseline import PRETRAINED_SHA256
from vehicle_detection.common import digest, file_hash, freeze_json, read_json, require_paths
from vehicle_detection.data import convert_labels, scan_bdd
from vehicle_detection.evaluation_data import validate_audit
from vehicle_detection.five_class import derive_labels
from vehicle_detection.preprocessing_experiments import (
    calibration_sample,
    candidates,
    confirmation_decision,
    development_sample,
    rank_candidates,
    run_comparison,
    validate_manifest,
)
from vehicle_detection.protocol import validate_protocol
from vehicle_detection.stress import converter_records
from vehicle_detection.stress_eval import SETTINGS

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
DATA_ROOT = ROOT / "data/bdd100k"
PROTOCOL_PATH = ROOT / "runs/issue5-v1/protocol.json"
SUBSETS_PATH = ROOT / "runs/issue6-v2/subsets.json"
POLICY_PATH = ROOT / "docs/issue6-v2/policy.json"
FIVE_ROOT = ROOT / "runs/preprocessing-five-v1"
TRAIN_CONVERSION = ROOT / "runs/issue6-v2/conversion/S10"
DEV_CONVERSION = ROOT / "runs/issue5-v1/conversion"
AUDIT_PATH = ROOT / "docs/round2-data-v2/audit.json"
MANIFEST_PATH = ROOT / "docs/round2-evaluation-v2/evaluation_sets.json"
CHECKPOINT = ROOT / "weights/yolo26n.pt"
OUTPUT = ROOT / "runs/round2-preprocessing-v2"
REPORT = ROOT / "docs/round2-preprocessing-v2"
TARGET_IMAGES = 256
ALLOW_INFERENCE = False
require_paths(
    data=DATA_ROOT,
    protocol=PROTOCOL_PATH,
    audit=AUDIT_PATH,
    scenarios=MANIFEST_PATH,
    v2=POLICY_PATH,
    checkpoint=CHECKPOINT,
)
for destination in (OUTPUT, REPORT):
    for protected in (
        DATA_ROOT,
        PROTOCOL_PATH.parent,
        SUBSETS_PATH.parent,
        FIVE_ROOT,
        TRAIN_CONVERSION,
        DEV_CONVERSION,
        AUDIT_PATH.parent,
        MANIFEST_PATH.parent,
        POLICY_PATH.parent,
        CHECKPOINT.parent,
    ):
        a, b = destination.resolve(), protected.resolve()
        if a == b or a in b.parents or b in a.parents:
            raise ValueError(f"Output overlaps preserved input: {protected}")
a, b = OUTPUT.resolve(), REPORT.resolve()
if a == b or a in b.parents or b in a.parents:
    raise ValueError("Run and report roots must be independent")
for destination in (OUTPUT, REPORT):
    if (
        destination.exists()
        and any(destination.iterdir())
        and not (destination / "plan.json").is_file()
    ):
        raise ValueError("Unknown output directory; choose a new root")
cv2.setNumThreads(8)
protocol = validate_protocol(read_json(PROTOCOL_PATH))
audit = validate_audit(
    read_json(AUDIT_PATH),
    protocol,
    roots={
        "data": DATA_ROOT,
        "five": FIVE_ROOT,
        "training_conversion": TRAIN_CONVERSION,
        "development_conversion": DEV_CONVERSION,
        "protocol": PROTOCOL_PATH.parent,
        "subsets": SUBSETS_PATH.parent,
        "policy": POLICY_PATH.parent,
        "report": AUDIT_PATH.parent,
    },
)
manifest = read_json(MANIFEST_PATH)
validate_manifest(manifest, protocol)
if manifest["data_audit_id"] != audit["audit_id"]:
    raise ValueError("Evaluation sets belong to another saved data audit")
v2 = validate_policy(read_json(POLICY_PATH), protocol)
if file_hash(CHECKPOINT) != PRETRAINED_SHA256:
    raise ValueError("Use the pinned pretrained YOLO26n")
configs = candidates()
pilots = {
    role: calibration_sample(protocol, manifest, "calibration_" + role, target_images=TARGET_IMAGES)
    for role in ("exploration", "confirmation")
}
# Installed fixtures can locate the package even when the notebook has been copied.
import vehicle_detection.preprocessing_experiments as experiments

code = Path(experiments.__file__).parent
transfer_sample = development_sample(protocol, manifest, target_images=TARGET_IMAGES)
plan = {
    "version": "issue30-plan-v2",
    "sample_target": TARGET_IMAGES,
    "opencv_threads": cv2.getNumThreads(),
    "development_transfer": {
        "pilot_id": transfer_sample["pilot_id"],
        "purpose": "one descriptive check; no retuning or reselection",
        "scenarios": [s["id"] for s in manifest["scenarios"]],
    },
    "manifest_id": manifest["manifest_id"],
    "audit_id": audit["audit_id"],
    "v2_policy_id": v2["policy_id"],
    "pilots": {k: p["pilot_id"] for k, p in pilots.items()},
    "candidates": configs,
    "numeric_versions": {"numpy": np.__version__, "opencv": cv2.__version__},
    "checkpoint_sha256": PRETRAINED_SHA256,
    "settings": SETTINGS,
    "implementation": {
        name: file_hash(code / name)
        for name in (
            "preprocessing_experiments.py",
            "preprocess.py",
            "stress.py",
            "stress_eval.py",
            "baseline.py",
            "evaluation_stats.py",
        )
    },
    "selection": "five-class AP; family-balanced mean; Skip wins ties",
    "confirmation": {
        "minimum_gain": 0,
        "max_clean_regression": 0.005,
        "max_synthetic_regression": 0.01,
        "max_class_regression": 0.02,
        "minimum_class_groups": 5,
        "comparators": ["skip", "frozen_v2"],
        "fallback_search": False,
    },
}
plan["plan_id"] = digest(plan)
freeze_json(OUTPUT / "plan.json", plan)
freeze_json(REPORT / "plan.json", plan)
records = None


def measure(names, role="exploration"):
    global records
    pilot = transfer_sample if role == "development" else pilots[role]
    converted = (
        FIVE_ROOT / "conversion/development"
        if role == "development"
        else OUTPUT / "conversion" / role / "five"
    )
    if role != "development" and ALLOW_INFERENCE and not (converted / "conversion.json").is_file():
        if records is None:
            records, _, _ = scan_bdd(DATA_ROOT, strict=False)
        six = converted.parent / "six"
        convert_labels(converter_records(pilot, protocol, records, DATA_ROOT), six)
        derive_labels(six, converted, {"tune": pilot["rows"]})
    result = run_comparison(
        protocol,
        manifest,
        pilot,
        DATA_ROOT,
        converted,
        OUTPUT / role,
        {name: configs[name] for name in dict.fromkeys(["skip", "frozen_v2", *names])},
        v2,
        CHECKPOINT,
        allow_inference=ALLOW_INFERENCE,
    )
    if result is None:
        print(f"{role}: missing native results; set ALLOW_INFERENCE=True to measure.")
        return None
    freeze_json(REPORT / f"{role}-{'-'.join(names)}.json", result)
    return pd.DataFrame(result)


print("Frozen plan:", plan["plan_id"])
print("Calibration images:", {k: len(p["rows"]) for k, p in pilots.items()})
print("Fixed development transfer images:", len(transfer_sample["rows"]))

<a id="individual-operators"></a>

## 2. Individual operators

Report native mAP50:95/mAP50, class support, preprocessing mean/p95 and actual decision distributions. Negative findings stay in the same evidence files as gains. Cost includes quality routing and filters, excluding source reads, corruption and JPEG encoding; native detector timing remains in each native result.

In [ ]:
single_names = [
    name for name, config in configs.items() if "steps" in config and len(config["steps"]) == 1
]
singles = measure(single_names)
if singles is not None:
    display(
        singles[
            [
                "candidate",
                "scenario",
                "mAP50_95",
                "mAP50",
                "full_class_support",
                "mean_ms",
                "p95_ms",
            ]
        ]
    )

<a id="composition-order"></a>

## 3. Composition and order

Six preregistered pairs test both orders of Gamma/CLAHE, Denoise/Gamma and Bilateral/Sharpen. No exhaustive product search and no automatic second filter. This stage uses exactly the same exploration IDs as singles and controls.

In [ ]:
sequence_names = [
    name for name, config in configs.items() if "steps" in config and len(config["steps"]) == 2
]
sequences = measure(sequence_names)
if sequences is not None:
    display(sequences[["candidate", "scenario", "mAP50_95", "mAP50", "mean_ms", "p95_ms"]])

<a id="adaptive-comparison"></a>

## 4. Adaptive comparison

Compare fixed treatments with two simple quality-only candidates: dark→Gamma and noise→Bilateral / dark→Gamma / otherwise Skip. Rules are first-match; no corruption-family/severity metadata enters decisions. Save subgroup AP/support, representative TP-loss IDs and decision frequencies from the same native matching files. These slices are descriptive, not multiple independent significance tests.

In [ ]:
routing_names = [name for name, config in configs.items() if "rules" in config]
routing = measure(routing_names)
exploration = None
if all(t is not None for t in (singles, sequences, routing)):
    exploration = pd.concat([singles, sequences, routing]).drop_duplicates(
        ["candidate", "scenario"]
    )
    display(exploration[["candidate", "scenario", "mAP50_95", "decisions"]])
    if exploration.full_class_support.all() and (exploration.minimum_class_groups >= 5).all():
        ranking = rank_candidates(exploration, manifest["scenarios"])
        display(pd.DataFrame(ranking))
        figure, ax = plt.subplots(figsize=(10, 5))
        table = pd.DataFrame(ranking).sort_values("score")
        ax.barh(table.candidate, table.score)
        ax.set_xlabel("Family-balanced native five-class mAP50:95")
        figure.tight_layout()
        figure.savefig(REPORT / "exploration_ranking.png")
        plt.show()
    else:
        print("Insufficient class/group support: evidence is a pilot; selection is blocked.")

<a id="selected-policy"></a>

## 5. Selected policy and confirmation

Lock the top exploration candidate and hashes **before** confirmation evaluation. Confirmation evaluates that candidate plus Skip/v2 once on separate calibration groups. It must match or exceed both controls' family-balanced AP, with no clean regression beyond 0.005 AP or synthetic-scenario regression beyond 0.01 AP (absolute 0–1 units). Every class also needs five groups and no class/scenario regression beyond 0.02 AP. These are engineering gates, not confidence intervals. A failed gate records failure and never tries a runner-up on confirmation. Historical v2 used calibration, so this is not a historically untouched holdout.

A successful config contains its full replay recipe, pinned code/dependency/evidence identities and hash. It is a candidate for downstream training; one fixed development transfer check follows confirmation and is descriptive only, with no repeated tuning and no final-test access.

In [ ]:
decision = None
transfer = None
if exploration is not None and (
    exploration.full_class_support.all() and (exploration.minimum_class_groups >= 5).all()
):
    ranking = rank_candidates(exploration, manifest["scenarios"])
    shortlist = {
        "plan_id": plan["plan_id"],
        "candidate": ranking[0]["candidate"],
        "ranking": ranking,
        "exploration_evidence_sha256": digest(exploration.to_dict("records")),
    }
    freeze_json(REPORT / "shortlist.json", shortlist)
    confirmation = measure([shortlist["candidate"]], role="confirmation")
    if confirmation is not None:
        evidence = {
            "plan": plan,
            "confirmation": digest(confirmation.to_dict("records")),
            "v2_recipes": v2["recipes"],
            "manifest_id": manifest["manifest_id"],
        }
        decision = confirmation_decision(
            confirmation, manifest["scenarios"], shortlist, configs, evidence
        )
        freeze_json(REPORT / "decision.json", decision)
        if decision["status"] == "selected_calibrated_candidate":
            freeze_json(REPORT / "policy.json", decision)
        display(decision)
        # This is a single fixed diagnostic after selection; it never feeds the shortlist.
        transfer = measure([shortlist["candidate"]], role="development")
        if transfer is not None:
            display(transfer[["candidate", "scenario", "mAP50_95", "mAP50", "mean_ms"]])
else:
    print("No selection: complete exploration with all five classes is required.")

<a id="summary"></a>

## 6. Summary and evidence

Keep native caches/exports under the run root; Git tracks compact plans, metrics and plots. Changed code, IDs, settings or candidates require a new root. This notebook never launches training or inspects the final holdout. After locking and confirming its candidate it measures one preregistered development transfer sample, with no retuning. Fixture and tiny smoke results never justify policy selection.

In [ ]:
summary = {
    "plan_id": plan["plan_id"],
    "status": decision["status"] if decision else "pending_detection_evidence",
    "selected_policy_id": (
        decision["policy_id"]
        if decision and decision["status"] == "selected_calibrated_candidate"
        else None
    ),
    "development_transfer": "completed_descriptive_no_retuning"
    if transfer is not None
    else "pending",
    "final_holdout": "sealed",
}
# Pending status may advance after opt-in inference; immutable evidence stays in plan/results.
print(summary)
if decision is not None:
    freeze_json(REPORT / "summary.json", summary)